# Análisis de calidad del aire

Práctica de selección, filtrado, limpieza y agregación con pandas. El código prioriza `calidad_aire_estaciones.csv`; en este proyecto solo está disponible `calidad_aire_estaciones.xlsx`, por lo que se usa ese archivo como alternativa. El Excel contiene fechas en algunas celdas de columnas de medición; se reportan y tratan como valores no numéricos faltantes, sin inventar mediciones.

In [9]:
from pathlib import Path

import pandas as pd

ruta_csv = Path("calidad_aire_estaciones.csv")
ruta_excel = Path("calidad_aire_estaciones.xlsx")

if ruta_csv.is_file():
    df_aire = pd.read_csv(ruta_csv, index_col=False)
    archivo_cargado = ruta_csv
elif ruta_excel.is_file():
    df_aire = pd.read_excel(ruta_excel, index_col=None)
    archivo_cargado = ruta_excel
else:
    raise FileNotFoundError(
        "No se encontró calidad_aire_estaciones.csv ni calidad_aire_estaciones.xlsx."
    )

# Quita columnas de índice exportadas accidentalmente como columnas sin nombre.
columnas_sin_nombre = df_aire.columns.astype(str).str.match(r"^Unnamed(?:: \d+)?$")
df_aire = df_aire.loc[:, ~columnas_sin_nombre]

columnas_medicion = [
    "Material_Particulado_PM25",
    "Nivel_Ozono_ppb",
    "Temperatura_C",
    "Velocidad_Viento_kmh",
]
for columna in columnas_medicion:
    valores_originales = df_aire[columna]
    valores_numericos = valores_originales.map(
        lambda valor: pd.to_numeric(valor, errors="coerce")
        if isinstance(valor, (str, int, float))
        else float("nan")
    )
    valores_invalidos = valores_originales.notna() & valores_numericos.isna()
    if valores_invalidos.any():
        print(f"{columna}: {int(valores_invalidos.sum())} valores no numéricos tratados como NaN")
    df_aire[columna] = valores_numericos

print(f"Archivo cargado: {archivo_cargado}")
print(f"Dimensiones: {df_aire.shape}")
print("Columnas:", df_aire.columns.tolist())
display(df_aire.head())
df_aire.info()

Material_Particulado_PM25: 210 valores no numéricos tratados como NaN
Nivel_Ozono_ppb: 251 valores no numéricos tratados como NaN
Temperatura_C: 889 valores no numéricos tratados como NaN
Velocidad_Viento_kmh: 805 valores no numéricos tratados como NaN
Archivo cargado: calidad_aire_estaciones.xlsx
Dimensiones: (1012, 8)
Columnas: ['ID_Estacion', 'Sector', 'Material_Particulado_PM25', 'Nivel_Ozono_ppb', 'Temperatura_C', 'Humedad_Relativa', 'Velocidad_Viento_kmh', 'Alerta_Ambiental']


,ID_Estacion,Sector,Material_Particulado_PM25,Nivel_Ozono_ppb,Temperatura_C,Humedad_Relativa,Velocidad_Viento_kmh,Alerta_Ambiental
0,EST-6001,Centro Histórico,NaN,NaN,NaN,60,NaN,Verde
1,EST-6002,Zona Industrial,48.0,NaN,24.0,86,NaN,Amarilla
2,EST-6003,Corredor Vial,NaN,NaN,NaN,51,33.5,Naranja
3,EST-6004,Corredor Vial,65.0,18.0,NaN,56,NaN,Verde
4,EST-6005,Centro Histórico,48.0,NaN,20.0,77,NaN,Amarilla


<class 'pandas.DataFrame'>
RangeIndex: 1012 entries, 0 to 1011
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID_Estacion                1012 non-null   str    
 1   Sector                     1012 non-null   str    
 2   Material_Particulado_PM25  731 non-null    float64
 3   Nivel_Ozono_ppb            690 non-null    float64
 4   Temperatura_C              123 non-null    float64
 5   Humedad_Relativa           1012 non-null   int64  
 6   Velocidad_Viento_kmh       207 non-null    float64
 7   Alerta_Ambiental           1012 non-null   str    
dtypes: float64(4), int64(1), str(3)
memory usage: 63.4 KB


## Selección, acceso por etiqueta y duplicados

Se conserva la primera fila repetida. La fila con etiqueta 175 se consulta antes de eliminar duplicados, sobre el DataFrame recién importado.

In [10]:
serie_pm25 = df_aire["Material_Particulado_PM25"]
tabla_sector_clima_alerta = df_aire[["Sector", "Temperatura_C", "Alerta_Ambiental"]]

print("Serie de PM2.5:")
display(serie_pm25)
print("Selección de varias columnas:")
display(tabla_sector_clima_alerta.head())

registro_175 = df_aire.loc[175]
print("Registro con etiqueta 175:")
display(registro_175)

cantidad_duplicados = int(df_aire.duplicated().sum())
df_aire = df_aire.drop_duplicates(keep="first")
print(f"Duplicados eliminados: {cantidad_duplicados}")
print(f"Dimensiones sin duplicados: {df_aire.shape}")

Serie de PM2.5:


0        NaN
1       48.0
2        NaN
3       65.0
4       48.0
        ... 
1007    22.0
1008    48.0
1009    22.0
1010     NaN
1011     NaN
Name: Material_Particulado_PM25, Length: 1012, dtype: float64

Selección de varias columnas:


,Sector,Temperatura_C,Alerta_Ambiental
0,Centro Histórico,NaN,Verde
1,Zona Industrial,24.0,Amarilla
2,Corredor Vial,NaN,Naranja
3,Corredor Vial,NaN,Verde
4,Centro Histórico,20.0,Amarilla


Registro con etiqueta 175:


ID_Estacion                         EST-6176
Sector                       Zona Industrial
Material_Particulado_PM25               48.0
Nivel_Ozono_ppb                         42.0
Temperatura_C                            NaN
Humedad_Relativa                          40
Velocidad_Viento_kmh                     NaN
Alerta_Ambiental                       Verde
Name: 175, dtype: object

Duplicados eliminados: 12
Dimensiones sin duplicados: (1000, 8)


## Filtrado de observaciones

In [11]:
pm25_critico = df_aire[df_aire["Material_Particulado_PM25"] > 55.0]
sectores_seleccionados = df_aire[
    df_aire["Sector"].isin(["Zona Industrial", "Corredor Vial"])
]
humedad_intervalo = df_aire[df_aire["Humedad_Relativa"].between(60, 80, inclusive="both")]
alerta_roja_viento_bajo = df_aire.query(
    "Alerta_Ambiental == 'Roja' and Velocidad_Viento_kmh < 10.0"
)
zona_ozono_alto = df_aire[
    df_aire["Sector"].str.startswith("Zona", na=False)
    & (df_aire["Nivel_Ozono_ppb"] > 40.0)
]

print("PM2.5 > 55.0:", len(pm25_critico), "registros")
display(pm25_critico)
print("Zona Industrial o Corredor Vial:", len(sectores_seleccionados), "registros")
display(sectores_seleccionados)
print("Humedad entre 60% y 80% inclusive:", len(humedad_intervalo), "registros")
display(humedad_intervalo)
print("Alerta Roja y viento < 10 km/h:", len(alerta_roja_viento_bajo), "registros")
display(alerta_roja_viento_bajo)
print("Sector iniciado por 'Zona' y ozono > 40.0:", len(zona_ozono_alto), "registros")
display(zona_ozono_alto)

PM2.5 > 55.0: 164 registros


,ID_Estacion,Sector,Material_Particulado_PM25,Nivel_Ozono_ppb,Temperatura_C,Humedad_Relativa,Velocidad_Viento_kmh,Alerta_Ambiental
3,EST-6004,Corredor Vial,65.0,18.0,NaN,56,NaN,Verde
5,EST-6006,Centro Histórico,65.0,NaN,NaN,44,34.1,Naranja
19,EST-6020,Zona Residencial,85.5,42.0,NaN,70,NaN,Naranja
20,EST-6021,Parque Natural,65.0,NaN,NaN,48,15.0,Verde
23,EST-6024,Zona Residencial,110.0,55.5,31.0,78,NaN,Amarilla
...,...,...,...,...,...,...,...,...
987,EST-6988,Parque Natural,85.5,NaN,NaN,51,NaN,Verde
989,EST-6990,Zona Residencial,65.0,NaN,NaN,47,NaN,Verde
990,EST-6991,Parque Natural,65.0,42.0,25.0,45,21.0,Amarilla
992,EST-6993,Zona Residencial,85.5,70.0,NaN,77,14.0,Verde


Zona Industrial o Corredor Vial: 400 registros


,ID_Estacion,Sector,Material_Particulado_PM25,Nivel_Ozono_ppb,Temperatura_C,Humedad_Relativa,Velocidad_Viento_kmh,Alerta_Ambiental
1,EST-6002,Zona Industrial,48.0,NaN,24.0,86,NaN,Amarilla
2,EST-6003,Corredor Vial,NaN,NaN,NaN,51,33.5,Naranja
3,EST-6004,Corredor Vial,65.0,18.0,NaN,56,NaN,Verde
6,EST-6007,Corredor Vial,35.5,42.0,31.2,66,29.2,Naranja
9,EST-6010,Corredor Vial,NaN,55.5,20.0,70,13.0,Naranja
...,...,...,...,...,...,...,...,...
981,EST-6982,Zona Industrial,NaN,42.0,NaN,78,NaN,Verde
988,EST-6989,Zona Industrial,NaN,18.0,NaN,87,NaN,Verde
993,EST-6994,Zona Industrial,65.0,42.0,NaN,79,16.0,Naranja
996,EST-6997,Zona Industrial,35.5,NaN,NaN,49,NaN,Verde


Humedad entre 60% y 80% inclusive: 386 registros


,ID_Estacion,Sector,Material_Particulado_PM25,Nivel_Ozono_ppb,Temperatura_C,Humedad_Relativa,Velocidad_Viento_kmh,Alerta_Ambiental
0,EST-6001,Centro Histórico,NaN,NaN,NaN,60,NaN,Verde
4,EST-6005,Centro Histórico,48.0,NaN,20.0,77,NaN,Amarilla
6,EST-6007,Corredor Vial,35.5,42.0,31.2,66,29.2,Naranja
7,EST-6008,Parque Natural,48.0,NaN,NaN,72,NaN,Roja
9,EST-6010,Corredor Vial,NaN,55.5,20.0,70,13.0,Naranja
...,...,...,...,...,...,...,...,...
984,EST-6985,Zona Residencial,NaN,NaN,NaN,78,12.0,Naranja
992,EST-6993,Zona Residencial,85.5,70.0,NaN,77,14.0,Verde
993,EST-6994,Zona Industrial,65.0,42.0,NaN,79,16.0,Naranja
994,EST-6995,Centro Histórico,48.0,42.0,NaN,70,NaN,Roja


Alerta Roja y viento < 10 km/h: 1 registros


,ID_Estacion,Sector,Material_Particulado_PM25,Nivel_Ozono_ppb,Temperatura_C,Humedad_Relativa,Velocidad_Viento_kmh,Alerta_Ambiental
670,EST-6671,Parque Natural,NaN,55.5,NaN,67,9.0,Roja


Sector iniciado por 'Zona' y ozono > 40.0: 171 registros


,ID_Estacion,Sector,Material_Particulado_PM25,Nivel_Ozono_ppb,Temperatura_C,Humedad_Relativa,Velocidad_Viento_kmh,Alerta_Ambiental
10,EST-6011,Zona Residencial,35.5,95.0,NaN,72,NaN,Verde
19,EST-6020,Zona Residencial,85.5,42.0,NaN,70,NaN,Naranja
22,EST-6023,Zona Residencial,NaN,42.0,NaN,57,NaN,Naranja
23,EST-6024,Zona Residencial,110.0,55.5,31.0,78,NaN,Amarilla
31,EST-6032,Zona Industrial,48.0,42.0,NaN,88,NaN,Verde
...,...,...,...,...,...,...,...,...
992,EST-6993,Zona Residencial,85.5,70.0,NaN,77,14.0,Verde
993,EST-6994,Zona Industrial,65.0,42.0,NaN,79,16.0,Naranja
995,EST-6996,Zona Residencial,22.0,55.5,NaN,52,NaN,Verde
998,EST-6999,Zona Industrial,22.0,42.0,NaN,71,NaN,Verde


## Imputación y clasificación de temperatura

Los límites de la categoría **Templado** incluyen 18 °C y 25 °C.

In [12]:
media_pm25 = df_aire["Material_Particulado_PM25"].mean()
df_aire["Material_Particulado_PM25"] = df_aire[
    "Material_Particulado_PM25"
].fillna(media_pm25)
df_aire["Nivel_Ozono_ppb"] = df_aire["Nivel_Ozono_ppb"].fillna(0.0)

def clasificar_temperatura(temperatura):
    if pd.isna(temperatura):
        return pd.NA
    if temperatura < 18:
        return "Frío"
    if temperatura <= 25:
        return "Templado"
    return "Cálido"

df_aire["Clima"] = df_aire["Temperatura_C"].apply(clasificar_temperatura)

print(f"Media usada para imputar PM2.5: {media_pm25:.2f} µg/m³")
print("Nulos restantes:", df_aire[["Material_Particulado_PM25", "Nivel_Ozono_ppb"]].isna().sum().to_dict())
display(df_aire[["Temperatura_C", "Clima"]].head())

Media usada para imputar PM2.5: 43.12 µg/m³
Nulos restantes: {'Material_Particulado_PM25': 0, 'Nivel_Ozono_ppb': 0}


,Temperatura_C,Clima
0,NaN,NaN
1,24.0,Templado
2,NaN,NaN
3,NaN,NaN
4,20.0,Templado


## Renombrado y agregaciones por sector

In [13]:
df_aire = df_aire.rename(
    columns={
        "Material_Particulado_PM25": "PM25",
        "Velocidad_Viento_kmh": "Viento_KMH",
    }
)
columnas_reordenadas = ["ID_Estacion", "Clima"] + [
    columna for columna in df_aire.columns if columna not in {"ID_Estacion", "Clima"}
]
df_aire = df_aire[columnas_reordenadas]

resumen_sector = df_aire.groupby("Sector").agg(
    PM25_promedio=("PM25", "mean"),
    Temperatura_maxima=("Temperatura_C", "max"),
)

print("Columnas iniciales tras reordenar:", df_aire.columns[:2].tolist())
display(resumen_sector)

Columnas iniciales tras reordenar: ['ID_Estacion', 'Clima']


,PM25_promedio,Temperatura_maxima
Sector,,
Centro Histórico,41.989755,31.4
Corredor Vial,43.670075,31.9
Parque Natural,43.338355,31.9
Zona Industrial,41.736669,31.4
Zona Residencial,44.499086,32.0


## Rango por alerta y agrupación multinivel

In [14]:
def rango(series):
    return series.max() - series.min()

dispersion_alerta = df_aire.groupby("Alerta_Ambiental").agg(
    {"PM25": rango, "Humedad_Relativa": rango}
)

promedios_sector_alerta = df_aire.groupby(["Sector", "Alerta_Ambiental"]).agg(
    PM25_promedio=("PM25", "mean"),
    Humedad_promedio=("Humedad_Relativa", "mean"),
)

print("Rango de PM2.5 y humedad por alerta:")
display(dispersion_alerta)
print("Promedios por sector y alerta:")
display(promedios_sector_alerta)

Rango de PM2.5 y humedad por alerta:


,PM25,Humedad_Relativa
Alerta_Ambiental,,
Amarilla,88.0,54
Naranja,88.0,55
Roja,88.0,55
Verde,88.0,55


Promedios por sector y alerta:


PM25_promedio  Humedad_promedio
Sector           Alerta_Ambiental                                 
Centro Histórico Amarilla              41.870185         66.305556
                 Naranja               43.702262         72.541667
                 Roja                  34.248744         58.727273
                 Verde                 42.386112         66.819672
Corredor Vial    Amarilla              43.996908         66.015385
                 Naranja               42.917216         70.615385
                 Roja                  49.612857         67.583333
                 Verde                 43.139015         68.336134
Parque Natural   Amarilla              45.103903         69.705882
                 Naranja               43.973925         69.972973
                 Roja                  43.129916         64.117647
                 Verde                 42.310173         69.420561
Zona Industrial  Amarilla              38.955677         68.000000
                 Naranja               42.358263         69.941176
                 Roja                  47.169285         62.750000
                 Verde                 42.275587         67.247525
Zona Residencial Amarilla              46.689058         70.488889
                 Naranja               47.151401         70.828571
                 Roja                  47.672698         76.000000
                 Verde                 42.109247         67.410000